In [1]:
# 패키지 설정
import pandas as pd
import numpy as np
import os
import time
import json
import warnings
import requests
from io import BytesIO
from tqdm import tqdm
import time
import os

# 저장 폴더 생성 
os.makedirs('data/raw/comtrade', exist_ok=True)
os.makedirs('data/output', exist_ok=True)


In [60]:
# SSL 검증 끄기 
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# API 키 설정 & URL 설정 
COMTRADE_API_KEY = "1d4e89d295bc4d97afa080099b5151bf"  # 본인 API 키를 넣어야 함 
BASE_URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

In [86]:
#--- 국가 코드 가져오기 함수 ---  
REFERENCE_URL = "https://comtrade.un.org/Data/cache/partnerAreas.json"

def get_all_partner_chunks(chunk_size= 30) : 
    try : 
        # 데이터 요청 코드 
        response = requests.get(REFERENCE_URL, verify=False, timeout=60)
        response.raise_for_status() # 에러시 바로 멈추기 

        # 데이터 제이슨으로 로드해서 디코드(BOM 제거) 
        data = json.loads(response.content.decode('utf-8-sig'))

        # data 안에서 results만 끄집어내기 
        df = pd.DataFrame(data['results'])

        # id 중 숫자만 있는 것만 끄집어 내기
        df = df[df["id"].astype(str).str.isnumeric()]
        # 숫자를 리스트화 
        partner_list = df['id'].astype(str).tolist()
        # 숫자를 chunk_size로 나누기 
        partner_chunks = [partner_list[i:i + chunk_size] for i in range(0, len(partner_list), chunk_size)]

        print(f"총 {len(partner_chunks)}개의 청크로 분할 완료\n")

        return partner_chunks
    
    except Exception as e:
        print(f"Partner 코드 수집 실패 : {e}")
        return None



In [87]:
partner_chunks = get_all_partner_chunks(chunk_size=30)

총 10개의 청크로 분할 완료



In [101]:
# --- 청크에 따른 데이터 모으기 ---- 
def fetch_by_partner_chunks(year, api_key, partner_chunks) : 
    
    all_chunk_data = []

    for i, chunk in enumerate(partner_chunks) : 
        partner_str = ",".join(map(str, chunk))

        params = {
            'reporterCode' : '410', 
            'partnerCode' : partner_str, 
            'flowCode' : 'X', 
            'period' : str(year), 
            'cmdCode' : 'AG6', 
            'maxRecords' : 100000,
            'includeDesc' : 'true', 
            'subscription-key' : api_key, 
        }

        try : 
            response = requests.get(
                BASE_URL, 
                params = params, 
                verify=False, 
                timeout=120
            )
            response.raise_for_status()
            data = response.json()

            if 'data' in data and len(data['data']) > 0 : 
                df = pd.DataFrame(data['data'])
                print(f"    청크 {i+1}/{len(partner_chunks)} : {len(df):,}행 수집 완료")
                all_chunk_data.append(df)
            else : 
                print(f"    청크 {i+1}/{len(partner_chunks)} : 데이터 없음")
        
        except Exception as e : 
            print(f"    청크 {i+1}/{len(partner_chunks)} 오류 : {e}")

        time.sleep(3)

    if all_chunk_data : 
        final_df = pd.concat(all_chunk_data, ignore_index=True)
        print(f"    {year}년 전체 병합 완료 : 총 {len(final_df):,} 행")
        return final_df
    else : 
        print(f"    {year}년 : 수집된 데이터가 없습니다.")
        return None

In [102]:
# ---- 데이터 모으기 실행 ---- 
START_YEAR = 2000
END_YEAR = 2023

if not partner_chunks : 
    print('국가 코드가 없습니다.')
else : 
    print('연도별 데이터 수집 시작합니다.')

    for year in range(START_YEAR, END_YEAR + 1) : 
        filepath = f'data/raw/comtrade/kr_export_hs6_{year}.csv'

        if os.path.exists(filepath) : 
            print(f'{year} 데이터가 이미 존재함, 건너뛰기')
            continue

        print(f'{year} 수집중...')

        df = fetch_by_partner_chunks(year, COMTRADE_API_KEY, partner_chunks) 

        if df is not None : 
            df.to_csv(filepath, index=False, encoding='utf-8-sig')
            print(f'    ▶ {year}년 데이터 저장 완료')
        time.sleep(3)
print('전체 수집 완료')

연도별 데이터 수집 시작합니다.
2000 수집중...
    청크 1/10 : 12,267행 수집 완료
    청크 2/10 : 11,600행 수집 완료
    청크 3/10 : 5,128행 수집 완료
    청크 4/10 : 5,370행 수집 완료
    청크 5/10 : 15,292행 수집 완료


KeyboardInterrupt: 